#### ***What is Re-ranking***
##### **Re-ranking is a technique that improves the ordering of retrieved chunks before send to the LLM**

#### **Why Re-ranking**
##### **Re-ranking decides the which retrived chunks are deserved to passed to the LLM.**

#### ***What is CrossEncoder***

##### ***The query and documents are processed Together.***

In [128]:
### Load the environment variables

from dotenv import load_dotenv

load_dotenv()

True

In [129]:
###path Exists
import os
path = "../kubernetes"

if os.path.exists(path):
    print("Path Is Exist.")
else:
    print("Path is not Exist.")

Path Is Exist.


In [106]:
### Load all pdf files using DirectoryLoader by PyMuPdfLoader

from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader

loader = DirectoryLoader(
    path,
    glob = "*.pdf",
    loader_cls=PyMuPDFLoader
)

documents = loader.load()


print("Number Of Documents:",len(documents))

Number Of Documents: 3983


In [107]:
### Chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap = 100
)

chunks = splitter.split_documents(documents)

print("Number Of Chunks:",len(chunks))

Number Of Chunks: 12694


In [108]:
### BM25 Retriever

from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(
    documents=chunks
)

In [109]:
bm25_retriever.k=10

In [110]:
###create a embedding by using langchain hugging face.

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
        model_name = "BAAI/bge-large-en-v1.5",
        model_kwargs = {"device":"cpu"},
        encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [111]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    persist_directory="../vectorstore/kubernetes_rag",
    collection_name="kubernetes_rag",
    embedding_function=embedding_model
)

In [112]:
similarity_retriever = vectorstore.as_retriever(search_type = "similarity",
        search_kwargs = {"k":10}
)

In [113]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[similarity_retriever,bm25_retriever],
    weights=[0.8,0.2]
)

In [114]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [115]:
query = "What is a Kubernetes Deployment?"

retrieved_chunks = hybrid_retriever.invoke(query,k=20)

pairs = [[query,chunk.page_content]
 for chunk in retrieved_chunks] ##Combining both query and retrieved_chunks

print(len(pairs))

20


In [116]:
scores = reranker.predict(pairs)
print(len(scores))

20


In [117]:
for chunk,score in zip(retrieved_chunks,scores):
    print("Score:",score)
    print("Page:",chunk.metadata.get("page"))
    print("Page Content:",chunk.page_content[:300])
    print("-"*60)

Score: 7.9213734
Page: 5
Page Content: system and its components. The Kubernetes control plane continually and actively manages
every object's actual state to match the desired state you supplied.
For example: in Kubernetes, a Deployment is an object that can represent an application
running on your cluster. When you create the Deploymen
------------------------------------------------------------
Score: 3.9102814
Page: 0
Page Content: The Concepts section helps you learn about the parts of the Kubernetes system and the
abstractions Kubernetes uses to represent your cluster, and helps you obtain a deeper
understanding of how Kubernetes works.
Overview
Kubernetes is a portable, extensible, open source platform for managing containe
------------------------------------------------------------
Score: 3.6074126
Page: 1
Page Content: Pods with higher Priority can schedule on Nodes. Eviction is the process of proactively
terminating one or more Pods on resource-starved Nodes.
Cluster Adminis

In [118]:
ranked_chunks = sorted(zip(retrieved_chunks,scores),key = lambda x:x[1], reverse=True)

for chunk in ranked_chunks[:5]:
    print(chunk)

(Document(id='8d5c7d19-a6b3-4b4f-b7e8-75ddb6c33e27', metadata={'file_path': '..\\kubernetes\\Concepts.pdf', 'format': 'PDF 1.7', 'creationdate': '', 'keywords': '', 'author': '', 'page': 5, 'subject': '', 'moddate': '', 'trapped': '', 'creator': '', 'total_pages': 676, 'modDate': '', 'creationDate': '', 'producer': 'WeasyPrint 56.1', 'source': '..\\kubernetes\\Concepts.pdf', 'title': ''}, page_content="system and its components. The Kubernetes control plane continually and actively manages\nevery object's actual state to match the desired state you supplied.\nFor example: in Kubernetes, a Deployment is an object that can represent an application\nrunning on your cluster. When you create the Deployment, you might set the Deployment spec\nto specify that you want three replicas of the application to be running. The Kubernetes system\nreads the Deployment spec and starts three instances of your desired application--updating the\nstatus to match your spec. If any of those instances should 

In [119]:
ranked_chunks

[(Document(id='8d5c7d19-a6b3-4b4f-b7e8-75ddb6c33e27', metadata={'file_path': '..\\kubernetes\\Concepts.pdf', 'format': 'PDF 1.7', 'creationdate': '', 'keywords': '', 'author': '', 'page': 5, 'subject': '', 'moddate': '', 'trapped': '', 'creator': '', 'total_pages': 676, 'modDate': '', 'creationDate': '', 'producer': 'WeasyPrint 56.1', 'source': '..\\kubernetes\\Concepts.pdf', 'title': ''}, page_content="system and its components. The Kubernetes control plane continually and actively manages\nevery object's actual state to match the desired state you supplied.\nFor example: in Kubernetes, a Deployment is an object that can represent an application\nrunning on your cluster. When you create the Deployment, you might set the Deployment spec\nto specify that you want three replicas of the application to be running. The Kubernetes system\nreads the Deployment spec and starts three instances of your desired application--updating the\nstatus to match your spec. If any of those instances should

In [120]:
top_chunks = [chunk for chunk,score in ranked_chunks[:5]]

In [121]:
query

'What is a Kubernetes Deployment?'

In [122]:
top_chunks

[Document(id='8d5c7d19-a6b3-4b4f-b7e8-75ddb6c33e27', metadata={'file_path': '..\\kubernetes\\Concepts.pdf', 'format': 'PDF 1.7', 'creationdate': '', 'keywords': '', 'author': '', 'page': 5, 'subject': '', 'moddate': '', 'trapped': '', 'creator': '', 'total_pages': 676, 'modDate': '', 'creationDate': '', 'producer': 'WeasyPrint 56.1', 'source': '..\\kubernetes\\Concepts.pdf', 'title': ''}, page_content="system and its components. The Kubernetes control plane continually and actively manages\nevery object's actual state to match the desired state you supplied.\nFor example: in Kubernetes, a Deployment is an object that can represent an application\nrunning on your cluster. When you create the Deployment, you might set the Deployment spec\nto specify that you want three replicas of the application to be running. The Kubernetes system\nreads the Deployment spec and starts three instances of your desired application--updating the\nstatus to match your spec. If any of those instances should 

In [123]:
test_queries = [
    "What is a Kubernetes Deployment?",
    "What is a Kubernetes Pod?",
    "What is a Kubernetes Service?",
    "What is a ReplicaSet?",
    "What is a ConfigMap?",
    "Deployment spec replicas desired state status",
    "Pod containers shared network storage node",
    "Service clusterIP selector endpoints Pods",
    "How is a Deployment related to a ReplicaSet?",
    "How does a ReplicaSet maintain Pods?"
]

In [124]:
for query in test_queries:

    retrieved_docs = hybrid_retriever.invoke(query)
    pairs = [[query,doc.page_content] for doc in retrieved_docs]
    print("Query:",query)
    scores = reranker.predict(pairs)

    ranked_docs = sorted(zip(retrieved_docs,scores),key = lambda x:x[1],reverse=True)

    top_5_chunks = ranked_docs[:5]

    for rank,(chunk,score) in enumerate(top_5_chunks,start=1):
        print(
            f"Rank {rank} | Page {chunk.metadata.get('page')} | Score {score}"
        )
    print("#"*50)

Query: What is a Kubernetes Deployment?
Rank 1 | Page 5 | Score 7.92137336730957
Rank 2 | Page 30 | Score 5.550075531005859
Rank 3 | Page 2 | Score 4.845337390899658
Rank 4 | Page 32 | Score 4.564247131347656
Rank 5 | Page 3 | Score 4.559100151062012
##################################################
Query: What is a Kubernetes Pod?
Rank 1 | Page 83 | Score 8.690617561340332
Rank 2 | Page 85 | Score 6.734971046447754
Rank 3 | Page 85 | Score 6.344106674194336
Rank 4 | Page 1 | Score 5.566908836364746
Rank 5 | Page 85 | Score 5.357826232910156
##################################################
Query: What is a Kubernetes Service?
Rank 1 | Page 246 | Score 7.262441635131836
Rank 2 | Page 449 | Score 6.653943061828613
Rank 3 | Page 0 | Score 6.529098987579346
Rank 4 | Page 3 | Score 6.149477958679199
Rank 5 | Page 36 | Score 5.7401580810546875
##################################################
Query: What is a ReplicaSet?
Rank 1 | Page 27 | Score 5.242628574371338
Rank 2 | Page 24 | Score

In [125]:
##Design a Prompt
from langchain_core.prompts import ChatPromptTemplate
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


prompt = ChatPromptTemplate.from_template("""
You are a Kubernetes documentation Assistant.

Answer the question using only the provided Context only.


Rules:
    1.Don't use information outside the Context.
    2.If the answer is not available in the context,
    say I don't have information based on the Provided Documents.

Context:
{context}

Question:
{question}

Answer:
""")

In [ ]:
### Create LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b"
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000184AEBB3CE0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000184AE76FB00>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [130]:
### Using the LLM to generate the response for every query using reranking and Hybrid
for query in test_queries:

    retrieved_docs = hybrid_retriever.invoke(query)
    pairs = [[query,doc.page_content] for doc in retrieved_docs]
    print("Query:",query)
    scores = reranker.predict(pairs)

    ranked_docs = sorted(zip(retrieved_docs,scores),key = lambda x:x[1],reverse=True)

    top_5_chunks = ranked_docs[:5]

    hybrid_docs = hybrid_retriever.invoke(query)[:5]

    rerank_context = "\n\n".join(doc.page_content for (doc,score) in top_5_chunks)
    hybrid_context = "\n\n".join(doc.page_content for doc in hybrid_docs)


    rerank_message = prompt.invoke({
        "question":query,
        "context":rerank_context
    })

    hybrid_message = prompt.invoke({
        "question":query,
        "context":hybrid_context
    })

    rerank_response = llm.invoke(rerank_message)
    hybrid_response = llm.invoke(hybrid_message)

    print("Rerank Response:",rerank_response)
    print("Hybrid Response:",hybrid_response)

    print("#"*50)

    

Query: What is a Kubernetes Deployment?
Rerank Response: content='A **Kubernetes Deployment** is a control‑plane object that represents an application running on the cluster.  \nWhen you create a Deployment you define a spec that describes the desired state—such as the number of replica Pods you want. The Kubernetes control plane continuously reads this spec, creates the required Pods, and updates the Deployment’s status to match the spec. If any of the Pods fail or the spec changes, the Deployment oversees the Pods, ensuring that the actual set of running Pods is brought back to the desired state. In short, a Deployment manages the lifecycle of the Pods that run your application.' additional_kwargs={'reasoning_content': 'We need to answer using only the provided context. The context includes a paragraph about Deployment: "For example: in Kubernetes, a Deployment is an object that can represent an application running on your cluster. When you create the Deployment, you might set the De